# BLP inversion with many agents per market using OneSlack formulation

In this notebook, I show how to use combRUM's OneSlack formulation to do a BLP inversion with $T$ markets and many agents $N_t$ in each market $t$. The OneSlack formulation performs well here because the number of master variables depends on the number of parameters, not on $N_t$.

In [1]:
import numpy as np
from linearmodels.iv import IV2SLS

import combrum as cb

n_obs = 5_000_000
N_SIMULATIONS = 1
N_MARKETS = 50
N_INSIDE_ITEMS = 10
N_COVARIATES = 3
N_PER_MARKET = n_obs // N_MARKETS
N_OBS = N_PER_MARKET * N_MARKETS


MAX_ITERATIONS = 5_000
TOLERANCE = 1e-3
SEED = 41
SHOCK_SCALE = 1.0
ALPHA_TRUE = 1.0
XI_LOADING = 0.5
XI_MEAN = 0.5

M = N_INSIDE_ITEMS + 1
K_LOCAL = N_COVARIATES + N_INSIDE_ITEMS
assert N_PER_MARKET * N_MARKETS == N_OBS

In [2]:
class MarketDemand(cb.Oracle, cb.FeatureMap):
    """One market: dense covariates plus local item fixed effects."""

    def __init__(self, *, x, est_shocks):
        self.x = np.asarray(x, dtype=np.float64)
        self.est_shocks = np.asarray(est_shocks, dtype=np.float64)
        self.N, self.J, self.C = map(int, self.x.shape)
        self.K = self.C + self.J
        self._rows = np.arange(self.N, dtype=np.int64)

    def _batch(self, agent_ids):
        ids = np.asarray(agent_ids, dtype=np.int64)
        if (
            self.est_shocks.shape[1] == 1
            and ids.size == self.N
            and np.array_equal(ids, self._rows)
        ):
            return ids, self.x, self.est_shocks[:, 0, :], self._rows
        obs = ids % self.N
        sim = ids // self.N
        return ids, self.x[obs], self.est_shocks[obs, sim], np.arange(ids.size)

    def price_batch(self, theta, local_ids):
        ids, x, eps, rows = self._batch(local_ids)
        inside_values = (
            np.einsum("ijk,k->ij", x, theta[: self.C], optimize=True)
            + theta[self.C :]
            + eps[:, 1:]
        )
        outside_values = eps[:, 0]
        best_inside = np.argmax(inside_values, axis=1)
        best_inside_values = inside_values[rows, best_inside]
        choose_inside = best_inside_values > outside_values
        best = np.where(choose_inside, best_inside + 1, 0).astype(np.int32)
        payoffs = np.where(choose_inside, best_inside_values, outside_values)
        return cb.DemandBatch.exact(ids, best, payoffs)

    def features_batch(
        self,
        ids,
        bundles,
        weights=None,
        aggregate=False,
    ):
        ids, x, eps, rows = self._batch(ids)
        bundles = np.asarray(bundles)
        if bundles.ndim == 1:
            chosen = bundles.astype(np.int64, copy=False)
        else:
            chosen = np.argmax(bundles, axis=1).astype(np.int64, copy=False)
        chosen_eps = eps[rows, chosen]
        inside_rows = np.flatnonzero(chosen > 0)

        if aggregate:
            if weights is None:
                raise ValueError("weights are required when aggregate=True")
            w = np.asarray(weights, dtype=np.float64)
            phi = np.zeros(self.K, dtype=np.float64)
            if inside_rows.size:
                items = chosen[inside_rows] - 1
                w_inside = w[inside_rows]
                phi[: self.C] = w_inside @ x[inside_rows, items]
                phi[self.C :] = np.bincount(
                    items,
                    weights=w_inside,
                    minlength=self.J,
                )
            return phi, float(np.dot(w, chosen_eps))

        Phi = np.zeros((ids.size, self.K), dtype=np.float64)
        if inside_rows.size:
            items = chosen[inside_rows] - 1
            Phi[inside_rows, : self.C] = x[inside_rows, items]
            Phi[inside_rows, self.C + items] = 1.0
        return Phi, chosen_eps

## Data

For each market $t=1,\ldots,T$ and item $j$, draw an excluded cost shifter $z_{tj}$, a demand/price unobservable $\xi_{tj}$ with nonzero mean, a price shock $\zeta^p_{tj}$, and a demand shock $\zeta^\delta_{tj}$. Then construct

$$p_{tj} = 0.7\, z_{tj} + \xi_{tj} + 0.3\, \zeta^p_{tj}$$
$$\delta_{tj} = -\alpha\, p_{tj} + 0.5\,\xi_{tj} + 0.5\, \zeta^\delta_{tj}.$$

$\xi_{tj}$ enters both price and utility, so price is endogenous. $z_{tj}$ shifts price directly and is excluded from utility, so it is the instrument in the price-sensitivity diagnostic.

Agent $i=1,\ldots,N_t$ in market $t$ chooses among the outside option and $J$ inside items:

$$u_{i0} = \varepsilon_{i0}, \qquad u_{ij} = X_{itj}'\beta + \delta_{tj} + \varepsilon_{ij}.$$

The observed choice is stored as the standard basis vector corresponding to $\arg\max\{u_{i0}, u_{i1},\ldots,u_{iJ}\}$.

In [3]:
rng = np.random.default_rng(SEED)

x = rng.normal(size=(N_MARKETS, N_PER_MARKET, N_INSIDE_ITEMS, N_COVARIATES))
beta_true = np.array([0.65, -0.35, 0.2], dtype=np.float64)
instruments = rng.normal(size=(N_MARKETS, N_INSIDE_ITEMS))
xi = XI_MEAN + rng.normal(size=(N_MARKETS, N_INSIDE_ITEMS))
price_noise = rng.normal(size=(N_MARKETS, N_INSIDE_ITEMS))
delta_noise = rng.normal(size=(N_MARKETS, N_INSIDE_ITEMS))
prices = 0.7 * instruments + xi + 0.3 * price_noise
delta_true = -ALPHA_TRUE * prices + XI_LOADING * xi + 0.5 * delta_noise

dgp_shocks = rng.normal(scale=SHOCK_SCALE, size=(N_MARKETS, N_PER_MARKET, M))
est_shocks = rng.normal(scale=SHOCK_SCALE, size=(N_MARKETS, N_PER_MARKET, N_SIMULATIONS, M))
observed_utilities = dgp_shocks.copy()
observed_utilities[:, :, 1:] += (
    np.einsum("tnjc,c->tnj", x, beta_true, optimize=True)
    + delta_true[:, None, :]
)
choice_labels = np.argmax(observed_utilities, axis=2)
observed = np.eye(M, dtype=np.float64)[choice_labels]

market_parameters = cb.Parameters(
    {
        "beta": (-3.0, 3.0, N_COVARIATES),
        "item": (-4.0, 4.0, N_INSIDE_ITEMS),
    }
)
{
    "observations_total": N_OBS,
    "markets": N_MARKETS,
    "observations_per_market": N_PER_MARKET,
    "alternatives_including_outside": M,
    "local_parameters": market_parameters.K,
}

{'observations_total': 5000000,
 'markets': 50,
 'observations_per_market': 100000,
 'alternatives_including_outside': 11,
 'local_parameters': 13}

## Market-Wise OneSlack formulation

For each market $t$, let $i=1,\ldots,N_t$ index agents. Let $\hat j_i \in \{0,1,\ldots,J\}$ denote the observed choice, with $0$ the outside option. Let $x_{ijk}$ be covariate $k$ for agent $i$ and inside item $j$. Set $x_{i0k}=0$ and $\delta_{t0}=0$. The parameters solved in market $t$ are $\beta_1,\ldots,\beta_K$ and $\delta_{t1},\ldots,\delta_{tJ}$.

With one simulation draw, the per-market OneSlack LP is

$$
\begin{aligned}
\min_{\beta,\,\delta_t,\,\bar u_t}\quad
& \bar u_t - \frac{1}{N_t}\sum_{i=1}^{N_t} 
\left(\sum_{k=1}^K x_{i\hat j_i k}\beta_k + \delta_{t\hat j_i}\right) \\
\text{s.t.}\quad
& \bar u_t \ge \frac{1}{N_t}\sum_{i=1}^{N_t} \left[
\left(\sum_{k=1}^K x_{ij_i k}\beta_k + \delta_{tj_i}\right)
+ \varepsilon_{tij_i}\right]
\qquad \forall (j_1,\ldots,j_{N_t}) \in \{0,1,\ldots,J\}^{N_t}.
\end{aligned}
$$

On the other hand, the NSlack LP is

$$
\begin{aligned}
\min_{\beta,\,\delta_t,\,u_{t1},\ldots,u_{tN_t}}\quad
& \frac{1}{N_t}\sum_{i=1}^{N_t} u_{ti}
- \frac{1}{N_t}\sum_{i=1}^{N_t}
\left(\sum_{k=1}^K x_{i\hat j_i k}\beta_k + \delta_{t\hat j_i}\right) \\
\text{s.t.}\quad
& u_{ti} \ge \sum_{k=1}^K x_{ijk}\beta_k + \delta_{tj} + \varepsilon_{tij}
\qquad \forall i=1,\ldots,N_t,\quad \forall j\in\{0,1,\ldots,J\}.
\end{aligned}
$$

The notebook uses OneSlack. Notice that the latter has an exponential number of constraints and a small number of variables. 

In [4]:
N_t = N_PER_MARKET * N_SIMULATIONS
problem = (
    ("T markets", N_MARKETS),
    ("N_t agents per market", N_t),
    ("J + 1 alternatives", M),
    ("K + J local parameters", K_LOCAL),
)
comparison = (
    (
        "OneSlack",
        f"K + J + 1 = {K_LOCAL + 1:,}",
        f"(J + 1)^N_t = {M}^{N_t} (about 10^{N_t * np.log10(M):,.0f})",
    ),
    (
        "NSlack",
        f"K + J + N_t = {K_LOCAL + N_t:,}",
        f"N_t (J + 1) = {N_t * M:,}",
    ),
)

print("problem size")
for label, value in problem:
    print(f"{label:<28} {value:>10,}")
print()
print("")
print(f"{'formulation':<13} {'variables':<24} constraints")
print("-" * 90)
for name, variables, constraints in comparison:
    print(f"{name:<13} {variables:<24} {constraints}")

problem size
T markets                            50
N_t agents per market           100,000
J + 1 alternatives                   11
K + J local parameters               13


formulation   variables                constraints
------------------------------------------------------------------------------------------
OneSlack      K + J + 1 = 14           (J + 1)^N_t = 11^100000 (about 10^104,139)
NSlack        K + J + N_t = 100,013    N_t (J + 1) = 1,100,000


In [5]:
def fit_market(t):
    oracle = MarketDemand(x=x[t], est_shocks=est_shocks[t])
    model = cb.Model(oracle, market_parameters, formulation=cb.OneSlack)
    data = cb.Data(
        observed_bundles=observed[t],
        shocks=est_shocks[t],
        observables=np.arange(N_PER_MARKET),
    )
    return cb.estimate(
        model,
        data,
        master_backend="highs",
        master_params={"u_lower_bound": None},
        max_iterations=MAX_ITERATIONS,
        tolerance=TOLERANCE,
        weights=np.full(
            N_PER_MARKET,
            1.0 / (N_PER_MARKET * N_SIMULATIONS),
            dtype=np.float64,
        ),
        activity=cb.ActivityConfig(
            label=f"t-{t:02d}",
            level="summary",
            stdout=True,
        ),
    )


market_fits = [fit_market(t) for t in range(N_MARKETS)]

[t-00] row generation: N=100000 S=1 K=13 agents=100000 ranks=1 tol=1.000e-03 max_iter=5000


[t-00] done converged=yes reason=converged iters=185 gap=9.956e-04 cuts=184 obj=1.151e+00 wall=1.45s


[t-01] row generation: N=100000 S=1 K=13 agents=100000 ranks=1 tol=1.000e-03 max_iter=5000


[t-01] done converged=yes reason=converged iters=189 gap=9.792e-04 cuts=188 obj=1.050e+00 wall=1.48s


[t-02] row generation: N=100000 S=1 K=13 agents=100000 ranks=1 tol=1.000e-03 max_iter=5000


...

[t-47] done converged=yes reason=converged iters=199 gap=8.920e-04 cuts=198 obj=9.778e-01 wall=1.67s


[t-48] row generation: N=100000 S=1 K=13 agents=100000 ranks=1 tol=1.000e-03 max_iter=5000


[t-48] done converged=yes reason=converged iters=192 gap=9.986e-04 cuts=191 obj=1.162e+00 wall=1.47s


[t-49] row generation: N=100000 S=1 K=13 agents=100000 ranks=1 tol=1.000e-03 max_iter=5000


[t-49] done converged=yes reason=converged iters=210 gap=8.267e-04 cuts=209 obj=1.173e+00 wall=1.71s


In [6]:
beta_hat = np.vstack([fit.theta_named()["beta"] for fit in market_fits])
delta_hat = np.vstack([fit.theta_named()["item"] for fit in market_fits])
delta_error = delta_hat - delta_true

t_rows = [
    {
        "t": t,
        "runtime_seconds": round(float(fit.runtime_seconds), 3),
        "iterations": int(fit.metadata["iterations"]),
        "active_cuts": int(fit.n_active_cuts),
        "converged": bool(fit.metadata["converged"]),
        "beta_max_abs_error": round(float(np.max(np.abs(beta_hat[t] - beta_true))), 4),
        "item_rmse": round(float(np.sqrt(np.mean(delta_error[t] ** 2))), 4),
    }
    for t, fit in enumerate(market_fits)
]

{
    "all_converged": all(row["converged"] for row in t_rows),
    "total_runtime_seconds": round(sum(row["runtime_seconds"] for row in t_rows), 3),
    "max_iterations_used": max(row["iterations"] for row in t_rows),
    "mean_beta_hat": beta_hat.mean(axis=0).round(6).tolist(),
    "beta_true": beta_true.round(6).tolist(),
    "beta_market_rmse": float(np.sqrt(np.mean((beta_hat - beta_true) ** 2))),
    "market_item_rmse": float(np.sqrt(np.mean(delta_error**2))),
}

{'all_converged': True,
 'total_runtime_seconds': 77.916,
 'max_iterations_used': 217,
 'mean_beta_hat': [0.649976, -0.349435, 0.20048],
 'beta_true': [0.65, -0.35, 0.2],
 'beta_market_rmse': 0.008907872584835587,
 'market_item_rmse': 0.07712268500097993}

## Price Sensitivity

Stack all market-item observations $(t, j)$. Regress the item effect on a constant and price $p_{tj}$, using the excluded cost shifter $z_{tj}$ as the instrument for price. The first diagnostic uses $\widehat\delta_{tj}$ from combRUM. The second uses the true $\delta_{tj}$ from the DGP.

In [7]:
def price_sensitivity_alpha(delta):
    delta_flat = np.asarray(delta, dtype=np.float64).ravel()
    price_flat = np.asarray(prices, dtype=np.float64).ravel()
    instrument_flat = np.asarray(instruments, dtype=np.float64).ravel()
    constant = np.ones(delta_flat.size, dtype=np.float64)
    ols = IV2SLS(
        delta_flat,
        np.column_stack([constant, price_flat]),
        None,
        None,
    ).fit()
    tsls = IV2SLS(delta_flat, constant, price_flat, instrument_flat).fit()
    return -float(ols.params.iloc[-1]), -float(tsls.params.iloc[-1])


ols_alpha, tsls_alpha = price_sensitivity_alpha(delta_hat)
true_ols_alpha, true_tsls_alpha = price_sensitivity_alpha(delta_true)

print("target alpha:", round(ALPHA_TRUE, 6))
print("ols alpha (estimated delta):", round(ols_alpha, 6))
print("2sls alpha (estimated delta):", round(tsls_alpha, 6))
print("ols alpha (true delta):", round(true_ols_alpha, 6))
print("2sls alpha (true delta):", round(true_tsls_alpha, 6))

target alpha: 1.0
ols alpha (estimated delta): 0.693956
2sls alpha (estimated delta): 0.930067
ols alpha (true delta): 0.699537
2sls alpha (true delta): 0.935589
